# Retrieval: hybrid search for RAG

Dense and sparse results fused with reciprocal rank fusion, filtered by kind, part and chapter.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
from tests.conftest import book_retriever
r = book_retriever('notebook')            # the synthetic book, in memory, with offline encoders
for h in r.search('Sonia read about Lazarus', kind='chunk', limit=3):
    print(round(h['score'], 3), h['label'], h['seq'], '|', h['text'][:60])

## Kinds: passages, quotes, summaries

In [ ]:
for kind in ('chunk', 'quote', 'summary'):
    best = r.search('Marmeladov tavern daughter', kind=kind, limit=1)
    print(kind, '->', best[0]['label'], '|', best[0]['text'][:80] if best else None)

## Narrowing to a part or chapter

In [ ]:
print([h['label'] for h in r.search('Raskolnikov', kind='chunk', part=1, limit=5)])
print([h['label'] for h in r.search('Raskolnikov', kind='chunk', part=1, chapter=2, limit=5)])

## Reciprocal rank fusion, by hand
Each list contributes 1 / (60 + rank); a passage high in both wins.

In [ ]:
dense_rank = ['a', 'b', 'c', 'd']
sparse_rank = ['c', 'a', 'e']
scores = {}
for ranking in (dense_rank, sparse_rank):
    for i, doc in enumerate(ranking, start=1):
        scores[doc] = scores.get(doc, 0) + 1 / (60 + i)
sorted(scores.items(), key=lambda x: -x[1])

## A real book (needs the stack and an indexed book)

In [ ]:
from marginalia.library import Library
from marginalia.index.retriever import Retriever
books = [b for b in Library().books() if b.status == 'ready']
if books:
    real = Retriever.for_book(books[0])
    print(books[0].title)
    for h in real.search('the first chapter', kind='summary', limit=3):
        print(round(h['score'], 3), h['label'], h['text'][:100])
else:
    print('no ready book in the library')